# Stage 5: One-time test-set evaluation

Thesis: *Explainable AI for Early Diabetes Prediction* (CDC BRFSS 2015). Config v1.0, master seed 42.

Implements Ch3 3.6.1-3.6.3, 3.7.1 (bullet 7), Table 3.2 and decisions **D2/D15, D7, D8, D17, S13**, with S8 (UTF-8), S10 (environment guard first) and S12e (bit-reproducibility).

The ten models, their thresholds, the best model and the two contrasts were frozen in Stage 4 (`frozen_models.json`). This notebook only **applies** them to the 50,736 test rows: nothing here is tuned, selected or adjusted, and no result may trigger retuning, recalibration or threshold revision (Ch3 3.7.1).

1. Checks the freeze (SHA-256), every pipeline, and the test partition (complement of the development rows; cross-checked against `split_dev_test.csv` and the repeated-profile flags).
2. Test probabilities for all ten models, stored once (`test_predictions.csv`); a rerun must reproduce them byte-for-byte.
3. Point estimates: AP, ROC-AUC; sensitivity, specificity, MCC, precision, F1, balanced accuracy, accuracy and predicted-positive rate at 0.50, the MCC threshold and the screening threshold; confusion matrices; calibration (Brier, log loss, calibration intercept and slope, reliability bins).
4. **2,000 paired, class-stratified bootstrap resamples** (seed `eval_bootstrap`), 95% percentile intervals, the same resamples for every model.
5. **Confirmatory contrasts** (Holm over four tests): CAT vs LR and CAT vs XGB, each by a paired bootstrap test of the AP difference and an exact McNemar test at the MCC thresholds. Other differences are exploratory intervals.
6. Exploratory: novel-profile sensitivity analysis (D2/D15) and the Sex / age-band table (D17).
7. Figures at 300 dpi and a report.

**Run:** confirm S13, set `S13_CONFIRMED = True` in Cell 2, put the notebook in `thesis_xai/`, Run All. Expected time on the PC: about 5-10 minutes.

In [1]:
# Cell 1 - environment guard (S10): runs before numpy / scikit-learn are imported.
import os
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/thesis_xai"   # edit if your Drive folder differs
else:
    PROJECT_DIR = os.path.abspath(".")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import env_setup
env_setup.ensure_pinned(PROJECT_DIR, install=IN_COLAB)

All pinned packages present at the pinned versions.


{}

In [2]:
# Cell 2 - configuration. The test set is used once: the run stops here until S13 is confirmed.
S13_CONFIRMED = True          # set to True only after S13 is confirmed in the decisions log
if not S13_CONFIRMED:
    raise SystemExit("Stage 5 uses the test set. Confirm S13 first, then set S13_CONFIRMED = True and run again.")

import hashlib
import json
import platform
import time
import warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
from scipy.stats import binomtest
from sklearn.metrics import average_precision_score, roc_auc_score
import statsmodels.api as sm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import thesis_common as tc

MACHINE = "colab" if IN_COLAB else "pc"
STAGE = "stage5_test_evaluation.ipynb"
FREEZE_SHA256 = "1dd5bf06708d148399675d9fc657857df1c86f015cbd83c24e79e8754b48c0d8"   # Stage 4 freeze (registry S4)
B = 2000                                   # Ch3 3.6.3
SEED = tc.SEEDS["eval_bootstrap"]          # S2
ALPHA = 0.05
N_BINS = 10                                # reliability diagram: equal-count bins
EPS = 1e-15                                # probability clipping for log loss and logit (S13)
OPS = [("0.50", "threshold_050"), ("MCC", "threshold_mcc"), ("screen", "threshold_screen")]
EXPECTED = {"n_raw": 253680, "test_rows": 50736, "test_pos": 7069, "profile_in_dev": 6919,
            "novel_rows": 43817, "novel_pos": 6744}

OUT = os.path.join(PROJECT_DIR, "stage5_outputs")
FIG = os.path.join(OUT, "figures")
os.makedirs(FIG, exist_ok=True)
assert SEED == 2705014006
print(f"machine={MACHINE}  python={platform.python_version()}  scikit-learn={sklearn.__version__}")

machine=pc  python=3.13.0  scikit-learn=1.6.1


In [3]:
# Cell 3 - the Stage 4 freeze and every frozen pipeline must be the canonical files.
fz_path = os.path.join(PROJECT_DIR, "stage4_outputs", "frozen_models.json")
if tc.sha256_file(fz_path) != FREEZE_SHA256:
    raise SystemExit("frozen_models.json is not the Stage 4 freeze. Restore it; do not evaluate.")
with open(fz_path, encoding="utf-8") as f:
    FZ = json.load(f)
MODELS = list(tc.MODELS)                   # all ten; do not change for the thesis run
BEST = FZ["best_model"]
CONTRASTS = {k: v for k, v in FZ["contrasts"].items() if v is not None and None not in v}
for m in MODELS:
    r = FZ["models"][m]
    for key, fk in [("pipeline_sha256", "pipeline"), ("oof_sha256", "oof_file")]:
        if tc.sha256_file(os.path.join(PROJECT_DIR, r[fk])) != r[key]:
            raise SystemExit(f"{m}: {r[fk]} differs from the freeze. Restore it; do not evaluate.")
print(f"Freeze verified. Best model {BEST}; contrasts {CONTRASTS}")
print(pd.DataFrame({m: {k: FZ['models'][m][k] for k in ['threshold_mcc', 'threshold_screen']} for m in MODELS}).T.to_string())

Freeze verified. Best model CAT; contrasts {'primary': ['CAT', 'LR'], 'secondary': ['CAT', 'XGB']}
      threshold_mcc  threshold_screen
LR         0.186139          0.125275
DT         0.623457          0.484708
RF         0.228568          0.140769
SVC        0.179207          0.122573
KNN        0.219124          0.143426
GNB        0.051547          0.033874
XGB        0.362178          0.238393
LGBM       0.388369          0.239523
CAT        0.550518          0.386190
MLP        0.208064          0.141574


In [4]:
# Cell 4 - test partition = all rows not in the development folds; cross-checked against two Stage 1 files.
ROW_ID_NAMES = ["row_id", "row", "idx", "index"]


def id_col(df, name):
    col = next((c for c in ROW_ID_NAMES if c in df.columns), None)
    if col is None:
        raise SystemExit(f"{name}: no row-id column ({ROW_ID_NAMES}); found {list(df.columns)}")
    return col


X_all, y_all = tc.load_data(PROJECT_DIR)              # SHA-256 checked
idx = tc.load_indices(PROJECT_DIR)                     # every index file checked against MANIFEST.csv
dev_rows = idx["dev_folds.csv"][id_col(idx["dev_folds.csv"], "dev_folds.csv")].to_numpy()
test_rows = np.setdiff1d(np.arange(len(X_all)), dev_rows)          # sorted raw-row positions
X_test = X_all.iloc[test_rows].reset_index(drop=True)
y = y_all[test_rows].astype(int)
n, P = len(y), int(y.sum())
N = n - P

# (a) split file: some two-valued column must mark exactly these rows
sp = idx["split_dev_test.csv"]
spr = sp[id_col(sp, "split_dev_test.csv")].to_numpy()
ok_split = any(np.array_equal(np.sort(spr[sp[c].to_numpy() == v]), test_rows)
               for c in sp.columns if c not in ROW_ID_NAMES and sp[c].nunique() == 2 for v in sp[c].unique())
# (b) repeated-profile flag, recomputed here and matched to the Stage 1 file by content (S3: 21 predictors)
dev_keys = pd.MultiIndex.from_frame(X_all.iloc[dev_rows].astype(int))
test_keys = pd.MultiIndex.from_frame(X_test.astype(int))
profile_in_dev = test_keys.isin(dev_keys)
fl = idx["test_repeated_profile_flags.csv"]
flr = fl[id_col(fl, "test_repeated_profile_flags.csv")].to_numpy()
ok_rows = np.array_equal(np.sort(flr), test_rows)
order = np.searchsorted(test_rows, flr) if ok_rows else None
ok_flag = ok_rows and any(np.array_equal(fl[c].to_numpy().astype(int)[np.argsort(order)], profile_in_dev.astype(int))
                          for c in fl.columns if c not in ROW_ID_NAMES and set(np.unique(fl[c])) <= {0, 1, True, False})
del idx, X_all, y_all

checks = {"test rows": (n, EXPECTED["test_rows"]), "test positives": (P, EXPECTED["test_pos"]),
          "split file marks the same test rows": (ok_split, True),
          "flag file covers the same test rows": (ok_rows, True),
          "recomputed profile flag equals a Stage 1 flag column": (ok_flag, True),
          "test rows with profile in development (S1-035)": (int(profile_in_dev.sum()), EXPECTED["profile_in_dev"]),
          "novel-profile rows (S1-037)": (int((~profile_in_dev).sum()), EXPECTED["novel_rows"]),
          "novel-profile positives (S1-038)": (int(y[~profile_in_dev].sum()), EXPECTED["novel_pos"])}
for k, (got, want) in checks.items():
    print(f"{'PASS' if got == want else 'FAIL'}  {k}: {got}")
if any(got != want for got, want in checks.values()):
    raise SystemExit("Test-partition checks failed; do not evaluate.")

PASS  test rows: 50736
PASS  test positives: 7069
PASS  split file marks the same test rows: True
PASS  flag file covers the same test rows: True
PASS  recomputed profile flag equals a Stage 1 flag column: True
PASS  test rows with profile in development (S1-035): 6919
PASS  novel-profile rows (S1-037): 43817
PASS  novel-profile positives (S1-038): 6744


In [5]:
# Cell 5 - test probabilities, computed once and stored (Ch3 3.7.2). A rerun must reproduce them exactly.
pred_path = os.path.join(OUT, "test_predictions.csv")
PROB = {}
for m in MODELS:
    pipe = joblib.load(os.path.join(PROJECT_DIR, FZ["models"][m]["pipeline"]))
    if m == "RF":
        pipe.set_params(clf__n_jobs=1)          # S12e: bit-stable RF probabilities
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=DeprecationWarning)
        p = pipe.predict_proba(X_test)[:, 1]    # SVC: calibrated probability (frozen CalibratedClassifierCV)
    if not (np.isfinite(p).all() and p.min() >= 0 and p.max() <= 1):
        raise SystemExit(f"{m}: test scores are not probabilities")
    PROB[m] = p
pred = pd.DataFrame({"row_id": test_rows, "y": y, "profile_in_dev": profile_in_dev.astype(int),
                     "Sex": X_test["Sex"].astype(int).to_numpy(), "Age": X_test["Age"].astype(int).to_numpy(),
                     **{f"p_{m}": PROB[m] for m in MODELS}})
blob = pred.to_csv(index=False, float_format="%.17g", lineterminator="\n").encode("utf-8")
if os.path.exists(pred_path):
    with open(pred_path, "rb") as f:
        if f.read() != blob:
            raise SystemExit("test_predictions.csv exists and differs from this run. It is never overwritten; report this.")
    print("test_predictions.csv: verified identical to the stored predictions")
else:
    with open(pred_path, "wb") as f:
        f.write(blob)
    print("test_predictions.csv: created")
back = pd.read_csv(pred_path, float_precision="round_trip")
assert all(np.array_equal(back[f"p_{m}"].to_numpy(), PROB[m]) for m in MODELS)
PRED_SHA = hashlib.sha256(blob).hexdigest()
print("sha256", PRED_SHA)

test_predictions.csv: created
sha256 b9813461cd03e4552b50c245c8750648d08b9c48c1960d249642982bc15114ed


In [6]:
# Cell 6 - metric functions. Weighted versions (bootstrap weights = resample counts) reproduce scikit-learn exactly.
class Curve:
    # Precomputed sort order and tie groups of one score vector; ap()/auc() accept per-row weights.
    def __init__(self, s, y):
        self.o = np.argsort(-s, kind="mergesort")
        ss = s[self.o]
        self.starts = np.r_[0, np.flatnonzero(np.diff(ss) != 0) + 1]
        self.y = y[self.o].astype(float)

    def _counts(self, w):
        ws = w[self.o]
        tp = np.cumsum(np.add.reduceat(ws * self.y, self.starts))
        fp = np.cumsum(np.add.reduceat(ws * (1 - self.y), self.starts))
        return tp, fp

    def ap(self, w):
        tp, fp = self._counts(w)
        rec = tp / tp[-1]
        den = tp + fp
        prec = np.divide(tp, den, out=np.zeros_like(tp), where=den > 0)   # zero-weight tie groups add nothing
        return float(np.sum(np.diff(np.r_[0.0, rec]) * prec))

    def auc(self, w):
        tp, fp = self._counts(w)
        return float(np.trapezoid(np.r_[0.0, tp / tp[-1]], np.r_[0.0, fp / fp[-1]]))


def from_counts(tp, fp, fn, tn):
    tp, fp, fn, tn = (np.asarray(v, float) for v in (tp, fp, fn, tn))
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    with np.errstate(invalid="ignore", divide="ignore"):
        prec = np.where(tp + fp > 0, tp / (tp + fp), np.nan)
        den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        mcc = np.where(den > 0, (tp * tn - fp * fn) / np.where(den > 0, den, 1), 0.0)
    tot = tp + fp + fn + tn
    return {"sensitivity": sens, "specificity": spec, "precision": prec, "f1": 2 * tp / (2 * tp + fp + fn),
            "mcc": mcc, "balanced_accuracy": (sens + spec) / 2, "accuracy": (tp + tn) / tot,
            "predicted_positive_rate": (tp + fp) / tot}


THR_METRICS = ["sensitivity", "specificity", "precision", "f1", "mcc", "balanced_accuracy", "accuracy", "predicted_positive_rate"]
CURVE = {m: Curve(PROB[m], y) for m in MODELS}
PC = {m: np.clip(PROB[m], EPS, 1 - EPS) for m in MODELS}
SQ = np.column_stack([(PROB[m] - y) ** 2 for m in MODELS])                          # Brier terms
LL = np.column_stack([-(y * np.log(PC[m]) + (1 - y) * np.log(1 - PC[m])) for m in MODELS])
# indicator columns: for each model x operating point, [pred & y, pred & ~y]
IND, COLS = [], []
for m in MODELS:
    for op, key in OPS:
        pr = PROB[m] >= FZ["models"][m][key]
        IND += [(pr & (y == 1)).astype(float), (pr & (y == 0)).astype(float)]
        COLS.append((m, op))
IND = np.column_stack(IND)


def all_metrics(w):
    # every metric for every model under row weights w (w = 1 gives the point estimate)
    W, Wp = w.sum(), (w * y).sum()
    Wn = W - Wp
    c = w @ IND
    out = {}
    for j, (m, op) in enumerate(COLS):
        tp, fp = c[2 * j], c[2 * j + 1]
        for k, v in from_counts(tp, fp, Wp - tp, Wn - fp).items():
            out[(m, op, k)] = float(v)
    br, ll = (w @ SQ) / W, (w @ LL) / W
    for i, m in enumerate(MODELS):
        out[(m, "-", "ap")] = CURVE[m].ap(w)
        out[(m, "-", "roc_auc")] = CURVE[m].auc(w)
        out[(m, "-", "brier")] = float(br[i])
        out[(m, "-", "log_loss")] = float(ll[i])
    return out


POINT = all_metrics(np.ones(n))
for m in MODELS:     # the weighted implementation must equal scikit-learn on the test set
    assert abs(POINT[(m, "-", "ap")] - average_precision_score(y, PROB[m])) < 1e-12, m
    assert abs(POINT[(m, "-", "roc_auc")] - roc_auc_score(y, PROB[m])) < 1e-12, m
print("metric functions verified against scikit-learn")

metric functions verified against scikit-learn


In [7]:
# Cell 7 - point estimates: confusion matrices and calibration (assessment only, 3.4.3)
cm_rows = []
for m in MODELS:
    for op, key in OPS:
        t = FZ["models"][m][key]
        pr = PROB[m] >= t
        cm_rows.append({"model": m, "operating_point": op, "threshold": t,
                        "tp": int((pr & (y == 1)).sum()), "fp": int((pr & (y == 0)).sum()),
                        "fn": int((~pr & (y == 1)).sum()), "tn": int((~pr & (y == 0)).sum())})
cm = pd.DataFrame(cm_rows)
cm.to_csv(os.path.join(OUT, "confusion_matrices.csv"), index=False, encoding="utf-8", float_format="%.10g")

cal_rows, rel_rows = [], []
for m in MODELS:
    n01 = int(((PROB[m] == 0) | (PROB[m] == 1)).sum())
    row = {"model": m, "n_prob_exactly_0_or_1": n01, "mean_predicted": PROB[m].mean(), "observed_rate": y.mean()}
    if n01 == 0:   # logistic recalibration needs finite logits (S13); otherwise reported as not estimable
        lg = np.log(PROB[m] / (1 - PROB[m]))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            sf = sm.GLM(y, sm.add_constant(lg), family=sm.families.Binomial()).fit()        # slope: logit(y) ~ a + b logit(p)
            itf = sm.GLM(y, np.ones((n, 1)), family=sm.families.Binomial(), offset=lg).fit()  # intercept: slope fixed at 1
        if sf.converged and itf.converged:
            (s_lo, s_hi), (i_lo, i_hi) = sf.conf_int()[1], itf.conf_int()[0]
            row.update({"calibration_slope": sf.params[1], "slope_ci_low": s_lo, "slope_ci_high": s_hi,
                        "calibration_intercept": itf.params[0], "intercept_ci_low": i_lo, "intercept_ci_high": i_hi})
    row.setdefault("calibration_slope", np.nan)
    row["slope_intercept_status"] = "estimated" if np.isfinite(row["calibration_slope"]) else \
        ("not estimable: probabilities of exactly 0 or 1" if n01 else "not estimable: fit did not converge")
    cal_rows.append(row)
    q = pd.qcut(pd.Series(PROB[m]).rank(method="first"), N_BINS, labels=False)            # equal-count bins
    for b_, g in pd.DataFrame({"p": PROB[m], "y": y, "bin": q}).groupby("bin"):
        rel_rows.append({"model": m, "bin": int(b_), "n": len(g), "mean_predicted": g.p.mean(), "observed_rate": g.y.mean()})
cal = pd.DataFrame(cal_rows)
rel = pd.DataFrame(rel_rows)
cal.to_csv(os.path.join(OUT, "calibration_slope_intercept.csv"), index=False, encoding="utf-8", float_format="%.10g")
rel.to_csv(os.path.join(OUT, "reliability_bins.csv"), index=False, encoding="utf-8", float_format="%.10g")
print(cal.round(4).to_string(index=False))

model  n_prob_exactly_0_or_1  mean_predicted  observed_rate  calibration_slope  slope_ci_low  slope_ci_high  calibration_intercept  intercept_ci_low  intercept_ci_high                         slope_intercept_status
   LR                      0          0.1386         0.1393             0.9914        0.9663         1.0164                 0.0077           -0.0202             0.0356                                      estimated
   DT                   2217          0.3746         0.1393                NaN           NaN            NaN                    NaN               NaN                NaN not estimable: probabilities of exactly 0 or 1
   RF                      0          0.1392         0.1393             1.0552        1.0275         1.0830                 0.0010           -0.0267             0.0286                                      estimated
  SVC                      0          0.1386         0.1393             0.9939        0.9690         1.0188                 0.0073          

In [8]:
# Cell 8 - 2,000 paired, class-stratified bootstrap resamples; the same resamples for every model.
pos_idx, neg_idx = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
KEYS = list(POINT.keys())
BOOT = np.empty((B, len(KEYS)))
rng = np.random.default_rng(SEED)
h = hashlib.sha256()
t0 = time.perf_counter()
for b in range(B):
    draw = np.r_[pos_idx[rng.integers(0, P, P)], neg_idx[rng.integers(0, N, N)]]   # positives, then negatives
    w = np.bincount(draw, minlength=n).astype(float)
    h.update(w.astype(np.int32).tobytes())
    with np.errstate(invalid="ignore", divide="ignore"):
        r = all_metrics(w)
    BOOT[b] = [r[k] for k in KEYS]
BOOT_HASH = h.hexdigest()
print(f"{B} resamples in {time.perf_counter() - t0:.0f} s; resample-weight hash {BOOT_HASH[:16]}")

rows = []
for j, (m, op, met) in enumerate(KEYS):
    col = BOOT[:, j]
    lo, hi = np.nanpercentile(col, [2.5, 97.5]) if np.isfinite(col).any() else (np.nan, np.nan)
    rows.append({"model": m, "operating_point": op, "metric": met, "estimate": POINT[(m, op, met)],
                 "ci_low": lo, "ci_high": hi, "n_undefined": int(np.isnan(col).sum())})
res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT, "test_metrics_ci.csv"), index=False, encoding="utf-8", float_format="%.10g")


def fmt(m, op, met, d=3):
    r = res[(res.model == m) & (res.operating_point == op) & (res.metric == met)].iloc[0]
    return f"{r.estimate:.{d}f} [{r.ci_low:.{d}f}, {r.ci_high:.{d}f}]"


main = pd.DataFrame([{"model": m, "AP": fmt(m, "-", "ap"), "ROC-AUC": fmt(m, "-", "roc_auc"),
                      "Sens@MCC": fmt(m, "MCC", "sensitivity"), "Spec@MCC": fmt(m, "MCC", "specificity"),
                      "MCC@MCC": fmt(m, "MCC", "mcc"), "Brier": fmt(m, "-", "brier", 4)} for m in MODELS])
main.to_csv(os.path.join(OUT, "test_main_table.csv"), index=False, encoding="utf-8")
pd.set_option("display.width", 220)
print(main.to_string(index=False))

2000 resamples in 60 s; resample-weight hash 5d84cf903a73cf26
model                   AP              ROC-AUC             Sens@MCC             Spec@MCC              MCC@MCC                   Brier
   LR 0.400 [0.390, 0.410] 0.821 [0.816, 0.825] 0.660 [0.649, 0.671] 0.799 [0.795, 0.803] 0.360 [0.351, 0.369] 0.0995 [0.0985, 0.1004]
   DT 0.400 [0.390, 0.409] 0.813 [0.808, 0.818] 0.630 [0.618, 0.641] 0.813 [0.809, 0.817] 0.355 [0.345, 0.364] 0.1808 [0.1790, 0.1827]
   RF 0.428 [0.417, 0.439] 0.826 [0.821, 0.830] 0.612 [0.601, 0.623] 0.833 [0.830, 0.837] 0.367 [0.357, 0.376] 0.0973 [0.0965, 0.0982]
  SVC 0.401 [0.391, 0.411] 0.821 [0.816, 0.826] 0.662 [0.651, 0.673] 0.798 [0.795, 0.802] 0.361 [0.352, 0.369] 0.0997 [0.0987, 0.1006]
  KNN 0.417 [0.406, 0.427] 0.823 [0.818, 0.827] 0.626 [0.615, 0.637] 0.820 [0.816, 0.823] 0.360 [0.351, 0.370] 0.0984 [0.0976, 0.0991]
  GNB 0.350 [0.341, 0.360] 0.796 [0.791, 0.801] 0.770 [0.761, 0.780] 0.689 [0.685, 0.694] 0.329 [0.322, 0.337] 0.1705 [0.1678, 0

In [9]:
# Cell 9 - confirmatory contrasts (Ch3 3.6.3, E3.20): paired bootstrap test of the AP difference and exact McNemar
# test at each model's MCC threshold; Holm-Bonferroni over the four tests. Other differences are exploratory.
KI = {k: j for j, k in enumerate(KEYS)}
DIFFS = [("-", "ap"), ("-", "roc_auc"), ("-", "brier"), ("-", "log_loss"),
         ("MCC", "mcc"), ("MCC", "sensitivity"), ("MCC", "specificity"),
         ("screen", "specificity"), ("screen", "sensitivity")]
diff_rows, tests = [], []
for name, (a, b_) in CONTRASTS.items():
    for op, met in DIFFS:
        d = BOOT[:, KI[(a, op, met)]] - BOOT[:, KI[(b_, op, met)]]
        est = POINT[(a, op, met)] - POINT[(b_, op, met)]
        lo, hi = np.percentile(d, [2.5, 97.5])
        diff_rows.append({"contrast": name, "model_a": a, "model_b": b_, "operating_point": op, "metric": met,
                          "difference_a_minus_b": est, "ci_low": lo, "ci_high": hi,
                          "role": "confirmatory" if (op, met) == ("-", "ap") else "exploratory"})
        if (op, met) == ("-", "ap"):
            if not np.isfinite(d).all():
                raise SystemExit("Non-finite bootstrap AP difference; report this.")
            p = min(1.0, 2 * min((d <= 0).sum() + 1, (d >= 0).sum() + 1) / (B + 1))
            tests.append({"contrast": name, "test": "paired bootstrap, AP difference", "statistic": est, "p_value": p})
    ca = (PROB[a] >= FZ["models"][a]["threshold_mcc"]).astype(int) == y
    cb = (PROB[b_] >= FZ["models"][b_]["threshold_mcc"]).astype(int) == y
    n_ab, n_ba = int((ca & ~cb).sum()), int((~ca & cb).sum())
    tests.append({"contrast": name, "test": f"exact McNemar at MCC thresholds ({a} right/{b_} wrong = {n_ab}, "
                  f"{a} wrong/{b_} right = {n_ba})", "statistic": n_ab - n_ba,
                  "p_value": binomtest(n_ab, n_ab + n_ba, 0.5).pvalue if n_ab + n_ba else 1.0})
diffs = pd.DataFrame(diff_rows)
tests = pd.DataFrame(tests, columns=["contrast", "test", "statistic", "p_value"])
# Holm-Bonferroni (step-down) over the confirmatory family
o = np.argsort(tests.p_value.to_numpy(), kind="mergesort")
k = len(tests)
adj = np.maximum.accumulate([min(1.0, (k - i) * tests.p_value.iloc[j]) for i, j in enumerate(o)]) if k else []
tests.loc[tests.index[o], "p_holm"] = adj
tests["reject_at_0.05"] = tests.p_holm < ALPHA
diffs.to_csv(os.path.join(OUT, "contrast_differences.csv"), index=False, encoding="utf-8", float_format="%.10g")
tests.to_csv(os.path.join(OUT, "confirmatory_tests.csv"), index=False, encoding="utf-8", float_format="%.10g")
print(tests.to_string(index=False))
print()
print(diffs.round(4).to_string(index=False))

 contrast                                                                                   test   statistic       p_value        p_holm  reject_at_0.05
  primary                                                        paired bootstrap, AP difference    0.031748  9.995002e-04  1.999000e-03            True
  primary  exact McNemar at MCC thresholds (CAT right/LR wrong = 2506, CAT wrong/LR right = 932) 1574.000000 1.674107e-164 6.696429e-164            True
secondary                                                        paired bootstrap, AP difference    0.001422  1.269365e-01  1.269365e-01           False
secondary exact McNemar at MCC thresholds (CAT right/XGB wrong = 878, CAT wrong/XGB right = 318)  560.000000  4.861404e-61  1.458421e-60            True

 contrast model_a model_b operating_point      metric  difference_a_minus_b  ci_low  ci_high         role
  primary     CAT      LR               -          ap                0.0317  0.0262   0.0377 confirmatory
  primary     CAT     

In [10]:
# Cell 10 - exploratory: novel-profile sensitivity analysis (D2, D15) and the Sex / age-band table (D17)
nov = ~profile_in_dev
yn = y[nov]
Pn, Nn = int(yn.sum()), int((1 - yn).sum())
pos_n, neg_n = np.flatnonzero(yn == 1), np.flatnonzero(yn == 0)
SUBCURVE = {m: Curve(PROB[m][nov], yn) for m in MODELS}


def sub_metrics(w):
    out = {}
    for m in MODELS:
        pr = PROB[m][nov] >= FZ["models"][m]["threshold_mcc"]
        tp, fp = w @ (pr & (yn == 1)), w @ (pr & (yn == 0))
        c = from_counts(tp, fp, (w * yn).sum() - tp, (w * (1 - yn)).sum() - fp)
        out.update({(m, "roc_auc"): SUBCURVE[m].auc(w), (m, "ap"): SUBCURVE[m].ap(w),
                    (m, "sensitivity"): float(c["sensitivity"]), (m, "specificity"): float(c["specificity"]),
                    (m, "mcc"): float(c["mcc"])})
    return out


SP = sub_metrics(np.ones(nov.sum()))
SK = list(SP.keys())
SB = np.empty((B, len(SK)))
rng2 = np.random.default_rng([SEED, 1])      # a separate, documented stream for the subset bootstrap
for b in range(B):
    draw = np.r_[pos_n[rng2.integers(0, Pn, Pn)], neg_n[rng2.integers(0, Nn, Nn)]]
    with np.errstate(invalid="ignore", divide="ignore"):
        r = sub_metrics(np.bincount(draw, minlength=nov.sum()).astype(float))
    SB[b] = [r[k] for k in SK]
srows = []
for j, (m, met) in enumerate(SK):
    full = POINT[(m, "-", met)] if met in ("roc_auc", "ap") else POINT[(m, "MCC", met)]
    lo, hi = np.percentile(SB[:, j], [2.5, 97.5])
    srows.append({"model": m, "metric": met, "full_test": full, "novel_profile": SP[(m, met)], "novel_ci_low": lo,
                  "novel_ci_high": hi, "difference_novel_minus_full": SP[(m, met)] - full})
sens_tab = pd.DataFrame(srows)
sens_tab["prevalence_full"] = P / n
sens_tab["prevalence_novel"] = Pn / nov.sum()
sens_tab.to_csv(os.path.join(OUT, "sensitivity_novel_profile.csv"), index=False, encoding="utf-8", float_format="%.10g")
print(f"Novel-profile subset: {nov.sum()} rows, prevalence {Pn / nov.sum():.4f} (full test {P / n:.4f}). AP is shown next to prevalence only (D15).")
print(sens_tab[sens_tab.metric != "ap"].pivot(index="model", columns="metric", values="difference_novel_minus_full").round(4).to_string())

age = X_test["Age"].astype(int).to_numpy()
groups = {"Sex = 0 (female)": X_test["Sex"].to_numpy() == 0, "Sex = 1 (male)": X_test["Sex"].to_numpy() == 1,
          "Age 18-44 (codes 1-5)": age <= 5, "Age 45-64 (codes 6-9)": (age >= 6) & (age <= 9),
          "Age 65+ (codes 10-13)": age >= 10}
grows = []
for m in MODELS:
    pr = PROB[m] >= FZ["models"][m]["threshold_mcc"]
    for gname, g in groups.items():
        yy, pp = y[g], pr[g]
        grows.append({"model": m, "group": gname, "n": int(g.sum()), "positives": int(yy.sum()),
                      "prevalence": yy.mean(), "sensitivity": pp[yy == 1].mean(), "specificity": 1 - pp[yy == 0].mean(),
                      "predicted_positive_rate": pp.mean()})
sub_tab = pd.DataFrame(grows)
sub_tab.to_csv(os.path.join(OUT, "subgroups_sex_age.csv"), index=False, encoding="utf-8", float_format="%.10g")
print(sub_tab[sub_tab.model.isin([BEST, "LR"])].round(3).to_string(index=False))

Novel-profile subset: 43817 rows, prevalence 0.1539 (full test 0.1393). AP is shown next to prevalence only (D15).
metric     mcc  roc_auc  sensitivity  specificity
model                                            
CAT    -0.0053  -0.0114       0.0174      -0.0228
DT     -0.0069  -0.0127       0.0177      -0.0273
GNB    -0.0109  -0.0155       0.0251      -0.0490
KNN    -0.0062  -0.0121       0.0201      -0.0277
LGBM   -0.0053  -0.0114       0.0179      -0.0231
LR     -0.0071  -0.0127       0.0167      -0.0282
MLP    -0.0053  -0.0115       0.0184      -0.0257
RF     -0.0047  -0.0117       0.0199      -0.0250
SVC    -0.0070  -0.0127       0.0188      -0.0296
XGB    -0.0056  -0.0115       0.0180      -0.0251
model                 group     n  positives  prevalence  sensitivity  specificity  predicted_positive_rate
   LR      Sex = 0 (female) 28482       3741       0.131        0.643        0.816                    0.244
   LR        Sex = 1 (male) 22254       3328       0.150        0.679

In [11]:
# Cell 11 - figures (300 dpi). Highlighted: the best model and the contrast models; the other models in grey.
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "figure.facecolor": "white", "axes.facecolor": "white"})
HL = [BEST] + [v[1] for v in CONTRASTS.values() if v[1] != BEST]
HUES = ["#2a78d6", "#eb6834", "#1baf7a"]          # validated categorical slots 1-3; distinct line styles as well
STYLE = {m: (HUES[i], ls) for i, (m, ls) in enumerate(zip(HL, ["-", "--", ":"]))}
GREY = "#b9b8b3"


def curve_points(m, kind):
    tp, fp = CURVE[m]._counts(np.ones(n))
    if kind == "pr":
        return np.r_[0.0, tp / tp[-1]], np.r_[1.0, tp / (tp + fp)]
    return np.r_[0.0, fp / fp[-1]], np.r_[0.0, tp / tp[-1]]


for kind, fname in [("pr", "fig_test_pr_curves.png"), ("roc", "fig_test_roc_curves.png")]:
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    for m in [x for x in MODELS if x not in HL]:
        a, b_ = curve_points(m, kind)
        ax.plot(a, b_, color=GREY, lw=0.9, zorder=1)
    for m in HL:
        a, b_ = curve_points(m, kind)
        met = "ap" if kind == "pr" else "roc_auc"
        ax.plot(a, b_, color=STYLE[m][0], ls=STYLE[m][1], lw=2, zorder=3,
                label=f"{tc.MODEL_NAMES[m]} ({'AP' if kind == 'pr' else 'AUC'} {POINT[(m, '-', met)]:.3f})")
    ax.plot([], [], color=GREY, lw=0.9, label=f"other models ({len(MODELS) - len(HL)})")
    if kind == "pr":
        ax.axhline(P / n, color="#52514e", ls=(0, (2, 2)), lw=1, label=f"prevalence {P / n:.3f}")
        ax.set_xlabel("Recall (sensitivity)"); ax.set_ylabel("Precision"); ax.set_ylim(0, 1)
    else:
        ax.plot([0, 1], [0, 1], color="#52514e", ls=(0, (2, 2)), lw=1, label="chance")
        ax.set_xlabel("False-positive rate (1 - specificity)"); ax.set_ylabel("True-positive rate (sensitivity)")
    ax.set_xlim(0, 1); ax.grid(color="#e6e5e0", lw=0.6); ax.legend(frameon=False, fontsize=7.5, loc="best")
    fig.tight_layout(); fig.savefig(os.path.join(FIG, fname), dpi=300); plt.close(fig)

fig, axes = plt.subplots(2, 5, figsize=(11, 4.8), sharex=True, sharey=True)
for ax, m in zip(axes.ravel(), MODELS):
    r = rel[rel.model == m]
    ax.plot([0, 1], [0, 1], color=GREY, lw=1)
    ax.plot(r.mean_predicted, r.observed_rate, color="#2a78d6", lw=2, marker="o", ms=4)
    ax.set_title(f"{m}  (Brier {POINT[(m, '-', 'brier')]:.4f})", fontsize=8.5)
    ax.grid(color="#e6e5e0", lw=0.6); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
for ax in axes[1]: ax.set_xlabel("Mean predicted probability")
for ax in axes[:, 0]: ax.set_ylabel("Observed positive rate")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_test_reliability.png"), dpi=300); plt.close(fig)

fig, axes = plt.subplots(2, 5, figsize=(11, 4.8))
for ax, m in zip(axes.ravel(), MODELS):
    r = cm[(cm.model == m) & (cm.operating_point == "MCC")].iloc[0]
    M = np.array([[r.tn, r.fp], [r.fn, r.tp]], float)
    R = M / M.sum(axis=1, keepdims=True)                                 # row-normalised (share of each true class)
    ax.imshow(R, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{int(M[i, j]):,}\n{R[i, j]:.1%}", ha="center", va="center", fontsize=7.5,
                    color="white" if R[i, j] > 0.55 else "#0b0b0b")
    ax.set_xticks([0, 1], ["Pred 0", "Pred 1"]); ax.set_yticks([0, 1], ["True 0", "True 1"])
    ax.set_title(f"{m}  (t = {r.threshold:.3f})", fontsize=8.5)
    for s in ax.spines.values(): s.set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_test_confusion_mcc.png"), dpi=300); plt.close(fig)
print("figures:", sorted(os.listdir(FIG)))

figures: ['fig_test_confusion_mcc.png', 'fig_test_pr_curves.png', 'fig_test_reliability.png', 'fig_test_roc_curves.png']


In [12]:
# Cell 12 - report and environment log
def write_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


env = tc.environment(MACHINE)
write_json({**env, "stage": STAGE, "written_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "pins_differ": {k: list(v) for k, v in tc.check_pins(PROJECT_DIR).items()},
            "freeze_sha256": FREEZE_SHA256, "test_predictions_sha256": PRED_SHA,
            "bootstrap": {"B": B, "seed": SEED, "weights_sha256": BOOT_HASH, "scheme": "class-stratified, positives then negatives"}},
           os.path.join(OUT, f"env_log_stage5_{MACHINE}.json"))
L = ["# Stage 5 report: one-time test-set evaluation", "",
     f"Generated by `{STAGE}` (config {tc.CONFIG_VERSION}) on {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC, machine {MACHINE}, "
     f"Python {env['python']}. Test set: {n:,} rows, {P:,} positive ({P / n:.2%}). Freeze {FREEZE_SHA256[:16]}; "
     f"predictions {PRED_SHA[:16]}. 95% intervals: {B} paired class-stratified bootstrap resamples (seed {SEED}).", "",
     "## Main results (MCC = frozen MCC-max threshold)", "", "| Model | AP | ROC-AUC | Sens | Spec | MCC | Brier |",
     "| --- | --- | --- | --- | --- | --- | --- |"]
L += [f"| {r.model} | {r.AP} | {r['ROC-AUC']} | {r['Sens@MCC']} | {r['Spec@MCC']} | {r['MCC@MCC']} | {r.Brier} |" for _, r in main.iterrows()]
L += ["", "## Calibration (assessment only)", "", "| Model | Calibration slope [95% CI] | Calibration intercept [95% CI] | Mean predicted / observed |",
      "| --- | --- | --- | --- |"]
for _, r in cal.iterrows():
    if r.slope_intercept_status == "estimated":
        L.append(f"| {r.model} | {r.calibration_slope:.3f} [{r.slope_ci_low:.3f}, {r.slope_ci_high:.3f}] | "
                 f"{r.calibration_intercept:.3f} [{r.intercept_ci_low:.3f}, {r.intercept_ci_high:.3f}] | {r.mean_predicted:.4f} / {r.observed_rate:.4f} |")
    else:
        L.append(f"| {r.model} | {r.slope_intercept_status} ({r.n_prob_exactly_0_or_1} rows) | - | {r.mean_predicted:.4f} / {r.observed_rate:.4f} |")
L += ["", f"## Confirmatory tests (Holm-Bonferroni over {len(tests)} tests)", "", "| Contrast | Test | Statistic | p | p (Holm) | Reject at 0.05 |",
      "| --- | --- | --- | --- | --- | --- |"]
L += [f"| {r.contrast} | {r.test} | {r.statistic:.4g} | {r.p_value:.4g} | {r.p_holm:.4g} | {r['reject_at_0.05']} |" for _, r in tests.iterrows()]
L += ["", f"Paired bootstrap p-values have resolution 2/(B+1) = {2 / (B + 1):.4f}; the smallest reportable value is p <= {2 / (B + 1):.4f}.",
      "", "Exploratory differences with intervals: `contrast_differences.csv`. Calibration: `calibration_slope_intercept.csv`, "
      "`reliability_bins.csv`. All metrics at all three operating points: `test_metrics_ci.csv`; counts: `confusion_matrices.csv`.",
      "Exploratory analyses: `sensitivity_novel_profile.csv` (D15), `subgroups_sex_age.csv` (D17).",
      "", "No result in this notebook may change a model, a threshold or the comparison plan (Ch3 3.7.1)."]
with open(os.path.join(OUT, "stage5_report.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(L) + "\n")
print("\n".join(L))
print("\nSend back: everything in stage5_outputs/ except test_predictions.csv (figures/*.png included).")

# Stage 5 report: one-time test-set evaluation

Generated by `stage5_test_evaluation.ipynb` (config v1.0) on 2026-10-14 05:35 UTC, machine pc, Python 3.13.0. Test set: 50,736 rows, 7,069 positive (13.93%). Freeze 1dd5bf06708d1483; predictions b9813461cd03e455. 95% intervals: 2000 paired class-stratified bootstrap resamples (seed 2705014006).

## Main results (MCC = frozen MCC-max threshold)

| Model | AP | ROC-AUC | Sens | Spec | MCC | Brier |
| --- | --- | --- | --- | --- | --- | --- |
| LR | 0.400 [0.390, 0.410] | 0.821 [0.816, 0.825] | 0.660 [0.649, 0.671] | 0.799 [0.795, 0.803] | 0.360 [0.351, 0.369] | 0.0995 [0.0985, 0.1004] |
| DT | 0.400 [0.390, 0.409] | 0.813 [0.808, 0.818] | 0.630 [0.618, 0.641] | 0.813 [0.809, 0.817] | 0.355 [0.345, 0.364] | 0.1808 [0.1790, 0.1827] |
| RF | 0.428 [0.417, 0.439] | 0.826 [0.821, 0.830] | 0.612 [0.601, 0.623] | 0.833 [0.830, 0.837] | 0.367 [0.357, 0.376] | 0.0973 [0.0965, 0.0982] |
| SVC | 0.401 [0.391, 0.411] | 0.821 [0.816, 0.826] | 0.662 [0.6